# 01 · Data and teacher embeddings (inference only)
Builds the MS MARCO data (`data.json`), audits it and checks for leakage, then embeds it with the frozen teacher.

The teacher (Qwen3-Embedding-4B) is **already trained and stays frozen**: this is embedding generation, not training.
It embeds the corpus and every query set once (MRL d=256, fp16) on 2 GPUs. These vectors are the fixed document index
and the targets every later notebook trains against.

How a text becomes a vector (`model.encode`): queries get the instruction prompt, passages none → tokenize with left
padding, end-of-text token appended → forward → last-token pooling → first 256 dims (MRL) → L2 normalise.
Batches are length-sorted; each GPU embeds its own shard; shards already on disk are skipped (resumable).

Input: none (first notebook). Output (attach to `02`, `03`, `04`): `cache/<mode>/{data.json, emb/teacher/*.npy}`, `runs/<mode>/metrics.json` (row `teacher`).

In [ ]:
# setup: clone/pull the repo on Kaggle, put it on sys.path, install what's missing (re-run to pull latest code)
import os, sys, importlib.util
REPO_URL = "https://github.com/pyther-hub/hlm-poc.git"
ON_KAGGLE = os.path.exists("/kaggle/working")
REPO = "/kaggle/working/hlm-poc" if ON_KAGGLE else os.path.abspath("..")
if ON_KAGGLE:
    if not os.path.exists(REPO):
        !git clone -q {REPO_URL} {REPO}
    else:
        !git -C {REPO} pull -q
sys.path.insert(0, REPO)

# HF token (higher rate limits, faster downloads). Kaggle: Add-ons -> Secrets -> add "HF_TOKEN", attach to notebook.
# Spawned workers inherit os.environ. Locally: export HF_TOKEN or `huggingface-cli login`.
if ON_KAGGLE and "HF_TOKEN" not in os.environ:
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception as e:
        print(f"HF_TOKEN secret not found ({type(e).__name__}): downloads stay unauthenticated")

import transformers
from packaging.version import Version
if Version(transformers.__version__) < Version("4.56"):  # Qwen3 + the dtype= kwarg
    !pip install -q -U "transformers>=4.56"
    print("transformers upgraded: restart the kernel, then re-run this cell")
for pkg in ("datasets", "pyarrow", "accelerate"):
    if importlib.util.find_spec(pkg) is None:
        !pip install -q {pkg}

%load_ext autoreload
%autoreload 2

## Control panel

In [ ]:
import os
import torch
from teacher import QUERY_PROMPT

DEBUG = True    # first pass: small data (~30k passages). Set False for the full run.
NPROCS = 2      # one process per T4

# --- paths. Later notebooks attach this one's output as an input (read only).
ROOT = "/kaggle/working" if ON_KAGGLE else REPO
MODE = "debug" if DEBUG else "full"
CACHE_DIR = f"{ROOT}/cache/{MODE}"
DATA_PATH = f"{CACHE_DIR}/data.json"
EMB_DIR = f"{CACHE_DIR}/emb/teacher"
RUN_DIR = f"{ROOT}/runs/{MODE}"

# --- data: queries from Tevatron/msmarco-passage train, distractors from Tevatron/msmarco-passage-corpus
DATA = dict(n_rows=210_000,     # first rows of the split (streamed, cached as parquet)
            core_n=100_000,     # seeded sample of queries -> train / val / test (split by query)
            n_val=5_000, n_test=5_000,
            n_align=200_000,    # alignment queries = train + other queries, never val/test
            n_distractors=500_000,  # random corpus passages added to the index (seeded with split_seed)
            split_seed=1234,
            use_title=False)
if DEBUG:
    DATA.update(n_rows=12_000, core_n=5_000, n_val=500, n_test=500, n_align=10_000, n_distractors=20_000)

TEACHER = dict(model="Qwen/Qwen3-Embedding-4B", dim=256, dtype="fp16",
               query_prompt=QUERY_PROMPT,          # queries only; passages get no prefix
               doc_max_len=128, query_max_len=64,  # tokens; query length includes the ~20-token prompt
               batch_size=64,                      # per GPU
               sets=("corpus", "align", "val", "test"), data_path=DATA_PATH, emb_dir=EMB_DIR)
CHECKS = dict(sanity_n=1000, timing_n=2000, budget_h=5.0)   # fp16 check / timing sample / reported budget

## 1 · Build data
Rows → split by query (train / val / test from a seeded core sample, the rest "other") → corpus = positives + one
stored negative of every core query, then `n_distractors` random passages of the full 8.8M corpus, deduplicated
→ alignment queries = train + other, never val/test.

In [ ]:
from data import build_data, load_distractors, load_json, load_rows, save_json

if os.path.exists(DATA_PATH):
    data = load_json(DATA_PATH)
    print("[data] cached")
else:
    rows = load_rows(DATA["n_rows"], CACHE_DIR)
    distractors = load_distractors(DATA["n_distractors"], DATA["split_seed"], CACHE_DIR)
    data = build_data(rows, distractors=distractors,
                      **{k: v for k, v in DATA.items() if k not in ("n_rows", "n_distractors")})
    save_json(data, DATA_PATH)
print({k: len(v["ids"]) for k, v in data.items()})

## 2 · Audit
What one training example looks like, positives per query, and text lengths in tokens (the `max_len` settings
truncate above them). `negative` is `negative_passages[0]` of the dataset: the "stored negative". It is **not
verified to be a BM25 negative**.

In [ ]:
print(data.keys())
print(len(data['corpus']['ids']), len(set(data['corpus']['ids'])))
print(len(data['align']['ids']), len(set(data['align']['ids'])))
print(len(data['train']['ids']), len(set(data['train']['ids'])))
print(len(data['val']['ids']), len(set(data['val']['ids'])))
print(len(data['test']['ids']), len(set(data['test']['ids'])))

In [ ]:
import numpy as np
from transformers import AutoTokenizer

corpus = data["corpus"]["texts"]
for i in range(3):
    print(f"query    {data['train']['texts'][i]}")
    print(f"positive {corpus[data['train']['pos'][i][0]][:200]}")
    print(f"negative {corpus[data['train']['neg'][i]][:200]}\n")

n_pos = np.array([len(p) for s in ("train", "val", "test") for p in data[s]["pos"]])
core_rows = {d for s in ("train", "val", "test") for ps, n in zip(data[s]["pos"], data[s]["neg"]) for d in (*ps, n)}
print(f"corpus {len(corpus)} passages: {len(core_rows)} from core queries, {len(corpus) - len(core_rows)} random distractors")
print(f"positives per query: mean {n_pos.mean():.2f}, max {n_pos.max()}, share with >1: {(n_pos > 1).mean():.3f}")

tok = AutoTokenizer.from_pretrained(TEACHER["model"])
for name, texts in (("query", data["train"]["texts"]), ("passage", corpus)):
    lens = np.array([len(x) for x in tok(texts[:2000])["input_ids"]])
    print(f"{name:<8} tokens p50 {np.percentile(lens, 50):.0f}  p95 {np.percentile(lens, 95):.0f}  max {lens.max()}")

## 3 · Leakage checks
Hard failures: a query in two of train/val/test, or a val/test query (by id or by normalised text) among the
alignment queries. Reported only: val/test positives that are also train positives. Passages are shared by design
(one corpus), so this is expected and not a leak of queries.

In [ ]:
ids = {s: set(data[s]["ids"]) for s in ("train", "val", "test", "align")}
text = {s: {t.strip().lower() for t in data[s]["texts"]} for s in ("val", "test", "align")}
assert not ids["train"] & ids["val"] and not ids["train"] & ids["test"] and not ids["val"] & ids["test"], "query in two splits"
assert not ids["align"] & (ids["val"] | ids["test"]), "val/test query id in align"
assert not text["align"] & (text["val"] | text["test"]), "val/test query text in align"
print("no query leakage: splits disjoint, align holds no val/test query (id or text)")

train_pos = {p for ps in data["train"]["pos"] for p in ps}
for s in ("val", "test"):
    shared = sum(any(p in train_pos for p in ps) for ps in data[s]["pos"])
    print(f"{s}: {shared}/{len(data[s]['pos'])} queries have a positive that is also a train positive")

## 4 · Pre-flight
fp16 vs fp32 on a passage sample (must have no NaNs), and a projection of how long the full run takes.

In [ ]:
from model import get_device
from teacher import fp16_sanity, free_gpu_memory, time_embedding

device = get_device()
if os.path.exists(f"{EMB_DIR}/corpus.npy"):
    print("corpus already embedded, pre-flight skipped")
else:
    sanity = fp16_sanity(TEACHER["model"], data["corpus"]["texts"][:CHECKS["sanity_n"]], TEACHER["dim"],
                         TEACHER["doc_max_len"], device, device_map_fp32="auto" if torch.cuda.device_count() > 1 else None)
    assert sanity["nan_rows"] == 0, "fp16 teacher produced NaNs"
    time_embedding(TEACHER, data, device, world_size=NPROCS, n_sample=CHECKS["timing_n"], budget_h=CHECKS["budget_h"])
    free_gpu_memory()

## 5 · Embed on both GPUs
Rank r embeds shard r of every set; rank 0 merges the shards into `<set>.npy` + `<set>_ids.json`.

In [ ]:
import time
from ddp import launch
from teacher import embed_main

t0 = time.time()
launch(embed_main, TEACHER, nprocs=NPROCS)
n_texts = sum(len(data[s]["ids"]) for s in TEACHER["sets"])
print(f"{n_texts} texts in {time.time() - t0:.0f} s ({n_texts / (time.time() - t0):.0f} texts/s, ~0 if already done)")

## 6 · Validate
Every set: shape `(N, 256)`, finite, unit norm, ids exactly `data.json`'s (so no record is missing or reordered).

In [ ]:
import numpy as np
from data import load_emb
from evaluation import check_emb

for name in TEACHER["sets"]:
    emb, ids = load_emb(EMB_DIR, name)
    assert emb.shape == (len(data[name]["ids"]), TEACHER["dim"]), f"{name}: shape {emb.shape}"
    check_emb(emb, ids, data[name]["ids"], name)   # finite, norms ~1, ids in data.json order
    norms = np.linalg.norm(emb.astype(np.float32), axis=1)
    print(f"{name:<7} {str(emb.shape):<15} {emb.dtype}  norm {norms.min():.4f}..{norms.max():.4f}  "
          f"{os.path.getsize(f'{EMB_DIR}/{name}.npy') / 1e6:8.1f} MB")

## 7 · Teacher ceiling
Teacher queries vs teacher corpus: the best any student can reach on this index. Row `teacher` in `metrics.json`.

In [ ]:
from evaluation import load_index, save_metrics, teacher_ceiling

docs, teacher_q = load_index(data, EMB_DIR, ("val", "test"))
rows = teacher_ceiling(data, docs, teacher_q, 100, device)
save_metrics(RUN_DIR, rows)
for split, m in rows["teacher"].items():
    print(split, {k: round(v, 4) for k, v in m.items()})